# Credit Card Fraud Detection

**Objective:** Handle severe class imbalance, compare classifiers and evaluate fraud detection using precision, recall, F1 and ROC-AUC.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score,roc_curve,precision_recall_fscore_support
df=pd.read_csv('transactions_synthetic.csv')
print(df.Is_Fraud.value_counts(normalize=True)*100)
sns.boxplot(data=df,x='Is_Fraud',y='Amount'); plt.yscale('log'); plt.title('Transaction amount by class'); plt.show()
sns.histplot(data=df,x='Hour',hue='Is_Fraud',multiple='stack'); plt.show()
X=df.drop(columns='Is_Fraud'); y=df.Is_Fraud
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
models={'Logistic Regression':make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=1000)),
        'Random Forest':RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42)}
for name,m in models.items():
    m.fit(Xtr,ytr); pred=m.predict(Xte); prob=m.predict_proba(Xte)[:,1]
    print(name,classification_report(yte,pred)); print('ROC-AUC:',roc_auc_score(yte,prob))
    sns.heatmap(confusion_matrix(yte,pred),annot=True,fmt='d'); plt.title(name+' confusion matrix'); plt.show()
    fpr,tpr,_=roc_curve(yte,prob); plt.plot(fpr,tpr,label=name)
plt.plot([0,1],[0,1],'k--'); plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.legend(); plt.title('ROC curves'); plt.show()
print('Recall is critical when missing fraud is costly, but precision controls false alarms and investigation workload. At scale, stream features, score transactions in low latency, monitor drift, and retrain regularly.')
